# LeetCode 1763 — Longest Nice Substring bằng Python

Link đề: https://leetcode.com/problems/longest-nice-substring/

Độ khó: **Easy** · Chủ đề: Hash Table, String, Divide and Conquer, Bit Manipulation, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def longestNiceSubstring(self, s: str) -> str:

```

Mục tiêu học:

1. Hiểu định nghĩa chuỗi **"nice"**: mọi chữ cái xuất hiện thì phải có **cả hoa lẫn thường**.
2. Duyệt **mọi chuỗi con** bằng hai vòng `i, j` và slicing `s[i:j+1]`; xử lý luật **"bằng độ dài thì lấy cái xuất hiện sớm nhất"**.
3. Dùng `set`, `str.swapcase`, `str.lower`/`str.upper` để kiểm tra "nice".
4. Tăng tốc kiểm tra bằng **bitmask** 26 bit (phép `|`, `<<`) — mở rộng chuỗi con từng ký tự một.
5. Học kỹ thuật **chia để trị** (divide and conquer): một ký tự "hỏng" không bao giờ nằm trong chuỗi nice → cắt chuỗi tại đó.
6. So sánh `O(n³)`, `O(n²)`, và chia để trị; tự giải các biến thể.

## 1. Bài toán yêu cầu gì?

Chuỗi `s` là **nice** nếu với **mỗi chữ cái** có trong `s`, nó xuất hiện ở **cả dạng hoa và dạng thường**.

- `"abABB"` nice: có `a` và `A`, có `b` và `B`.
- `"abA"` không nice: có `b` nhưng không có `B`.

Cho `s`, trả về **chuỗi con nice dài nhất**. Nếu nhiều chuỗi cùng độ dài, trả chuỗi **xuất hiện sớm nhất**. Không có thì trả `""`.

| s | Kết quả | Giải thích |
|---|---|---|
| `"YazaAay"` | `"aAa"` | chỉ chữ `a/A`, đủ cả hoa và thường |
| `"Bb"` | `"Bb"` | cả chuỗi là nice |
| `"c"` | `""` | không có chuỗi con nice |

Constraints: `1 <= s.length <= 100`; `s` gồm chữ cái tiếng Anh hoa và thường.

**Chuỗi con (substring)** là đoạn **liên tiếp**, khác với dãy con (subsequence).

In [1]:
s = "YazaAay"
for i, ch in enumerate(s):
    print(i, ch)
print("s[3:6] =", s[3:6])   # 'aAa'

0 Y
1 a
2 z
3 a
4 A
5 a
6 y
s[3:6] = aAa


## 2. Nền tảng Python cần biết

### 2.1. Slicing `s[i:j+1]` — chuỗi con từ `i` đến `j` (bao gồm)

`s[a:b]` lấy từ chỉ số `a` đến **trước** `b`. Muốn lấy cả `s[j]` thì viết `s[i:j+1]`.

In [2]:
s = "abcde"
print(s[1:3], s[1:3 + 1], s[0:len(s)])

bc bcd abcde


### 2.2. Liệt kê mọi chuỗi con

Hai vòng lặp: `i` là điểm bắt đầu, `j` là điểm kết thúc (`j >= i`). Chuỗi dài `n` có `n·(n+1)/2` chuỗi con.

In [3]:
s = "abc"
subs = [s[i:j + 1] for i in range(len(s)) for j in range(i, len(s))]
print(subs, "-> số lượng:", len(subs), "= 3*4/2")

['a', 'ab', 'abc', 'b', 'bc', 'c'] -> số lượng: 6 = 3*4/2


### 2.3. `set` và đổi hoa/thường

- `set(t)`: tập các ký tự khác nhau trong `t`.
- `ch.lower()`, `ch.upper()`: đổi sang thường/hoa.
- `ch.swapcase()`: đổi ngược kiểu chữ (`'a'` → `'A'`, `'B'` → `'b'`).
- `ch.islower()`, `ch.isupper()`: kiểm tra kiểu chữ.

In [4]:
t = "abABB"
print(set(t))
print('a'.upper(), 'B'.lower(), 'a'.swapcase(), 'B'.swapcase())
print('a'.islower(), 'A'.isupper())

{'a', 'A', 'B', 'b'}
A b A b
True True


### 2.4. Kiểm tra một chuỗi có nice không

Chuỗi `t` nice ⇔ với mỗi `ch` trong `set(t)`, `ch.swapcase()` cũng nằm trong `set(t)`.

In [5]:
def is_nice(t: str) -> bool:
    chars = set(t)
    return all(ch.swapcase() in chars for ch in chars)

for t in ["abABB", "abA", "aAa", "Bb", "c", ""]:
    print(repr(t), "->", is_nice(t))

'abABB' -> True
'abA' -> False
'aAa' -> True
'Bb' -> True
'c' -> False
'' -> True


Lưu ý: chuỗi rỗng `""` cho `True` (không có chữ cái nào để vi phạm). Đề trả `""` khi không có chuỗi con nice **khác rỗng**, nên ta chỉ cần khởi tạo đáp án là `""`.

### 2.5. Bitmask: dùng một số nguyên làm "tập 26 chữ cái"

Bit thứ `k` bằng 1 nghĩa là "có chữ cái thứ `k`" (`a=0, b=1, …, z=25`).

- `1 << k`: số chỉ có bit `k` bật. Ví dụ `1 << 2 = 4 = 0b100`.
- `mask | (1 << k)`: bật thêm bit `k`.
- `bin(x)`: xem dạng nhị phân.

In [6]:
k = ord('c') - ord('a')
print("k =", k, "| 1 << k =", 1 << k, "=", bin(1 << k))
mask = 0
for ch in "acd":
    mask |= 1 << (ord(ch) - ord('a'))
print("mask của 'acd' =", bin(mask))

k = 2 | 1 << k = 4 = 0b100
mask của 'acd' = 0b1101


Với chuỗi `t`, giữ hai mask: `lower` (chữ thường đã gặp) và `upper` (chữ hoa đã gặp). `t` nice ⇔ **`lower == upper`** — mỗi chữ cái có thường thì có hoa và ngược lại.

In [7]:
def masks(t):
    lower = upper = 0
    for ch in t:
        if ch.islower():
            lower |= 1 << (ord(ch) - ord('a'))
        else:
            upper |= 1 << (ord(ch) - ord('A'))
    return lower, upper

for t in ["abABB", "abA", "aAa"]:
    lo, up = masks(t)
    print(f"{t:<6} lower={bin(lo):<6} upper={bin(up):<6} nice? {lo == up}")

abABB  lower=0b11   upper=0b11   nice? True
abA    lower=0b11   upper=0b1    nice? False
aAa    lower=0b1    upper=0b1    nice? True


## 3. Cách trực giác — thử mọi chuỗi con

Gợi ý của đề: "brute force, kiểm tra từng chuỗi con".

1. `best = ""`.
2. Với mọi `i <= j`: `t = s[i:j+1]`; nếu `is_nice(t)` và `len(t) > len(best)` thì `best = t`.
3. Trả `best`.

Vì sao `>` (lớn hơn **hẳn**)? Ta duyệt `i` tăng dần, nên chuỗi cùng độ dài mà bắt đầu sớm hơn được gặp trước. Dùng `>` giữ lại chuỗi sớm nhất; dùng `>=` sẽ bị chuỗi sau ghi đè.

In [8]:
def longest_nice_brute(s: str) -> str:
    best = ""
    n = len(s)
    for i in range(n):
        for j in range(i, n):
            t = s[i:j + 1]
            if len(t) > len(best) and is_nice(t):
                best = t
    return best

print(repr(longest_nice_brute("YazaAay")))
print(repr(longest_nice_brute("Bb")))
print(repr(longest_nice_brute("c")))
print(repr(longest_nice_brute("dDzeE")))   # 'dD' và 'eE' cùng dài 2 -> lấy 'dD'

'aAa'
'Bb'
''
'dD'


Ta đặt `len(t) > len(best)` **trước** `is_nice(t)`: nhờ đánh giá ngắn mạch (short-circuit) của `and`, nếu chuỗi không dài hơn thì bỏ qua luôn bước kiểm tra tốn kém.

## 4. Nút thắt và insight

**Nút thắt của brute force:** có `~n²/2` chuỗi con, mỗi chuỗi tạo slice + `set` tốn `O(n)` → `O(n³)`. Với `n = 100` là ~170.000 phép — vẫn qua, nhưng ta lặp lại công việc: `s[i:j+1]` và `s[i:j+2]` chung gần hết ký tự.

**Insight 1 (mở rộng dần + bitmask):** cố định `i`, tăng `j` từng bước, **cập nhật** hai mask `lower/upper` bằng một phép `|` thay vì dựng lại `set`. Kiểm tra nice = so hai số nguyên → `O(1)`. Tổng `O(n²)`.

**Insight 2 (chia để trị):** nếu một ký tự `ch` của `s` **không có bạn** (`ch.swapcase()` không xuất hiện ở đâu trong `s`), thì **không chuỗi con nice nào chứa `ch`** — vì chuỗi con chỉ có ít ký tự hơn `s`. Vậy mọi chuỗi con nice nằm hẳn **bên trái** hoặc **bên phải** `ch` → cắt `s` tại `ch` và giải hai phần.

In [9]:
s = "YazaAay"
chars = set(s)
for i, ch in enumerate(s):
    print(f"i={i} {ch!r}: bạn {ch.swapcase()!r} có trong s? {ch.swapcase() in chars}")

i=0 'Y': bạn 'y' có trong s? True
i=1 'a': bạn 'A' có trong s? True
i=2 'z': bạn 'Z' có trong s? False
i=3 'a': bạn 'A' có trong s? True
i=4 'A': bạn 'a' có trong s? True
i=5 'a': bạn 'A' có trong s? True
i=6 'y': bạn 'Y' có trong s? True


Chỉ có `'z'` không có bạn (`'Y'` vẫn có bạn `'y'` ở cuối chuỗi). Vậy đáp án nằm trong `"Ya"` (bên trái `z`) hoặc `"aAay"` (bên phải `z`).

Điều thú vị: sau khi cắt, ký tự **trước đây có bạn** có thể **mất bạn**. Trong `"Ya"` thì `'Y'` không còn `'y'`; trong `"aAay"` thì `'y'` không còn `'Y'`. Ta lại cắt tiếp — đó là đệ quy.

## 5. Lời giải chính — chia để trị

### 5.1. Trường hợp cơ sở

Chuỗi có độ dài `< 2` không thể nice (cần ít nhất một cặp hoa/thường) → trả `""`.

### 5.2. Tìm ký tự "hỏng" đầu tiên và cắt

In [10]:
def find_bad(s: str) -> int:
    chars = set(s)
    for i, ch in enumerate(s):
        if ch.swapcase() not in chars:
            return i
    return -1          # không có ký tự hỏng -> s là nice

print(find_bad("YazaAay"), find_bad("aAay"), find_bad("aAa"))

2 3 -1


### 5.3. Ghép kết quả hai bên — giữ luật "sớm nhất"

`left` nằm trước `right` trong `s`. Nếu hai bên dài bằng nhau thì chọn `left`. Vì vậy dùng `len(left) >= len(right)`.

### 5.4. Hàm hoàn chỉnh

In [11]:
def longest_nice_dc(s: str) -> str:
    if len(s) < 2:
        return ""
    chars = set(s)
    for i, ch in enumerate(s):
        if ch.swapcase() not in chars:              # ký tự hỏng: không chuỗi nice nào chứa nó
            left = longest_nice_dc(s[:i])
            right = longest_nice_dc(s[i + 1:])
            return left if len(left) >= len(right) else right
    return s                                         # không có ký tự hỏng -> cả s là nice

for t in ["YazaAay", "Bb", "c", "dDzeE", "abABB"]:
    print(repr(t), "->", repr(longest_nice_dc(t)))

'YazaAay' -> 'aAa'
'Bb' -> 'Bb'
'c' -> ''
'dDzeE' -> 'dD'
'abABB' -> 'abABB'


**Đệ quy** (recursion): hàm gọi lại chính nó với input **nhỏ hơn** (`s[:i]`, `s[i+1:]` đều ngắn hơn `s`), nên chắc chắn dừng ở trường hợp cơ sở `len(s) < 2`.

## 6. TRACE từng bước

Cây đệ quy cho `"YazaAay"`:

```text
f("YazaAay")   set = {Y, a, z, A, y}; ký tự hỏng đầu tiên: 'z' (i=2)
├── f("Ya")    set = {Y, a}; 'Y' hỏng (i=0)
│   ├── f("")  -> ""
│   └── f("a") -> ""   (độ dài < 2)
│   => ""
└── f("aAay")  set = {a, A, y}; 'y' hỏng (i=3)
    ├── f("aAa") không có ký tự hỏng -> "aAa"
    └── f("")    -> ""
    => "aAa"
=> chọn giữa "" và "aAa" -> "aAa"
```

Hàm trace in đúng cây này, thụt lề theo độ sâu.

In [12]:
def trace_dc(s: str, depth: int = 0) -> str:
    pad = "    " * depth
    if len(s) < 2:
        print(f"{pad}f({s!r}) -> '' (độ dài < 2)")
        return ""
    chars = set(s)
    for i, ch in enumerate(s):
        if ch.swapcase() not in chars:
            print(f"{pad}f({s!r}): ký tự hỏng {ch!r} tại i={i} -> cắt thành {s[:i]!r} | {s[i+1:]!r}")
            left = trace_dc(s[:i], depth + 1)
            right = trace_dc(s[i + 1:], depth + 1)
            res = left if len(left) >= len(right) else right
            print(f"{pad}=> so {left!r} vs {right!r} -> {res!r}")
            return res
    print(f"{pad}f({s!r}): không có ký tự hỏng -> {s!r} là nice")
    return s

trace_dc("YazaAay")

f('YazaAay'): ký tự hỏng 'z' tại i=2 -> cắt thành 'Ya' | 'aAay'
    f('Ya'): ký tự hỏng 'Y' tại i=0 -> cắt thành '' | 'a'
        f('') -> '' (độ dài < 2)
        f('a') -> '' (độ dài < 2)
    => so '' vs '' -> ''
    f('aAay'): ký tự hỏng 'y' tại i=3 -> cắt thành 'aAa' | ''
        f('aAa'): không có ký tự hỏng -> 'aAa' là nice
        f('') -> '' (độ dài < 2)
    => so 'aAa' vs '' -> 'aAa'
=> so '' vs 'aAa' -> 'aAa'


'aAa'

### 6.1. Trace trường hợp hòa độ dài — luật "sớm nhất"

`"dDzeE"`: cắt tại `'z'` → hai bên `"dD"` và `"eE"` dài bằng nhau → chọn `left = "dD"`.

In [13]:
trace_dc("dDzeE")

f('dDzeE'): ký tự hỏng 'z' tại i=2 -> cắt thành 'dD' | 'eE'
    f('dD'): không có ký tự hỏng -> 'dD' là nice
    f('eE'): không có ký tự hỏng -> 'eE' là nice
=> so 'dD' vs 'eE' -> 'dD'


'dD'

### 6.2. Trace cách bitmask với `i` cố định

Với `s = "YazaAay"` và `i = 3`, mở rộng `j` dần: thấy mask thay đổi và lúc nào `lower == upper`.

In [14]:
def trace_mask_from(s: str, i: int) -> None:
    lower = upper = 0
    for j in range(i, len(s)):
        ch = s[j]
        if ch.islower():
            lower |= 1 << (ord(ch) - ord('a'))
        else:
            upper |= 1 << (ord(ch) - ord('A'))
        print(f"j={j} thêm {ch!r}: lower={lower:>27b} upper={upper:>27b} | {s[i:j+1]!r:<8} nice? {lower == upper}")

trace_mask_from("YazaAay", 3)

j=3 thêm 'a': lower=                          1 upper=                          0 | 'a'      nice? False
j=4 thêm 'A': lower=                          1 upper=                          1 | 'aA'     nice? True
j=5 thêm 'a': lower=                          1 upper=                          1 | 'aAa'    nice? True
j=6 thêm 'y': lower=  1000000000000000000000001 upper=                          1 | 'aAay'   nice? False


`f"{x:b}"` in số `x` ở dạng nhị phân; `:>27b` căn phải trong 27 ô để các cột thẳng hàng.

## 7. Code nộp LeetCode

In [15]:
class Solution:
    def longestNiceSubstring(self, s: str) -> str:
        if len(s) < 2:
            return ""
        chars = set(s)
        for i, ch in enumerate(s):
            if ch.swapcase() not in chars:
                left = self.longestNiceSubstring(s[:i])
                right = self.longestNiceSubstring(s[i + 1:])
                return left if len(left) >= len(right) else right
        return s

sol = Solution()
print(repr(sol.longestNiceSubstring("YazaAay")), repr(sol.longestNiceSubstring("Bb")), repr(sol.longestNiceSubstring("c")))

'aAa' 'Bb' ''


Chú ý: trong `class`, hàm đệ quy gọi lại chính nó qua `self.longestNiceSubstring(...)`.

## 8. Các lời giải khác

### 8.1. Brute force — bản nộp `O(n³)`

In [16]:
class SolutionBrute:
    def longestNiceSubstring(self, s: str) -> str:
        best = ""
        n = len(s)
        for i in range(n):
            for j in range(i + 1, n):          # độ dài >= 2
                if j - i + 1 > len(best):
                    t = s[i:j + 1]
                    chars = set(t)
                    if all(c.swapcase() in chars for c in chars):
                        best = t
        return best

print(repr(SolutionBrute().longestNiceSubstring("YazaAay")))

'aAa'


### 8.2. Mở rộng dần + bitmask — `O(n²)`

Cố định `i`, tăng `j`, cập nhật `lower`/`upper` bằng `|`. So sánh `lower == upper` là `O(1)`.

In [17]:
class SolutionBitmask:
    def longestNiceSubstring(self, s: str) -> str:
        n = len(s)
        best_i, best_len = 0, 0
        for i in range(n):
            lower = upper = 0
            for j in range(i, n):
                ch = s[j]
                if ch.islower():
                    lower |= 1 << (ord(ch) - ord('a'))
                else:
                    upper |= 1 << (ord(ch) - ord('A'))
                if lower == upper and j - i + 1 > best_len:
                    best_i, best_len = i, j - i + 1
        return s[best_i:best_i + best_len]

for t in ["YazaAay", "Bb", "c", "dDzeE"]:
    print(repr(t), "->", repr(SolutionBitmask().longestNiceSubstring(t)))

'YazaAay' -> 'aAa'
'Bb' -> 'Bb'
'c' -> ''
'dDzeE' -> 'dD'


Mẹo nhỏ: lưu `best_i, best_len` thay vì chuỗi — chỉ cắt chuỗi **một lần** ở cuối.

### 8.3. Ý tưởng sliding window theo số chữ cái khác nhau (tham khảo)

Với mỗi `k = 1..26`, tìm chuỗi con dài nhất có **đúng `k` chữ cái khác nhau** (không phân biệt hoa/thường) và cả `k` chữ đều đủ hoa + thường — dùng cửa sổ trượt hai con trỏ. Tổng `O(26 · n)`. Cách này hữu ích khi `n` rất lớn; với `n ≤ 100` thì chia để trị hoặc bitmask đã đủ và dễ viết hơn nhiều.

In [18]:
def longest_nice_window(s: str) -> str:
    best_i, best_len = 0, 0
    n = len(s)
    for k in range(1, 27):
        cnt = {}                     # ký tự (giữ hoa/thường) -> số lần trong cửa sổ
        letters = {}                 # chữ cái (lower) -> số lần trong cửa sổ
        left = 0
        for right in range(n):
            ch = s[right]
            cnt[ch] = cnt.get(ch, 0) + 1
            letters[ch.lower()] = letters.get(ch.lower(), 0) + 1
            while len(letters) > k:  # quá k chữ cái -> thu hẹp trái
                c = s[left]
                cnt[c] -= 1
                if cnt[c] == 0:
                    del cnt[c]
                letters[c.lower()] -= 1
                if letters[c.lower()] == 0:
                    del letters[c.lower()]
                left += 1
            if len(letters) == k and len(cnt) == 2 * k:   # đủ hoa + thường cho cả k chữ
                L = right - left + 1
                if L > best_len or (L == best_len and left < best_i):
                    best_i, best_len = left, L
    return s[best_i:best_i + best_len]

for t in ["YazaAay", "Bb", "c", "dDzeE"]:
    print(repr(t), "->", repr(longest_nice_window(t)))

'YazaAay' -> 'aAa'
'Bb' -> 'Bb'
'c' -> ''
'dDzeE' -> 'dD'


### 8.4. Bảng so sánh

Gọi `n = len(s)`, `Σ = 52` (số ký tự có thể).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---:|---:|---|
| Brute force | mọi chuỗi con + `set` | `O(n³)` | `O(n)` | Dễ nhất, đúng gợi ý đề |
| Mở rộng + bitmask | cập nhật `lower/upper` | `O(n²)` | `O(1)` | Nhanh, ít lỗi |
| Chia để trị | cắt tại ký tự hỏng | `O(n²)` xấu nhất, thường nhanh hơn | `O(n)` stack đệ quy (+ slice) | **Lời giải chính**, tư duy hay |
| Sliding window theo k | 26 lần hai con trỏ | `O(26 · n)` | `O(Σ)` | Mở rộng cho n lớn |

## 9. Vì sao lời giải đúng?

**Bổ đề:** nếu `ch` xuất hiện trong `s` nhưng `ch.swapcase()` không xuất hiện ở đâu trong `s`, thì mọi chuỗi con nice của `s` đều **không chứa** vị trí đó.
*Lý do:* chuỗi con `t` chứa `ch` thì để nice phải chứa `ch.swapcase()`; nhưng mọi ký tự của `t` đều lấy từ `s`, mà `s` không có `ch.swapcase()`. Mâu thuẫn.

**Hệ quả:** mọi chuỗi con nice nằm hoàn toàn trong `s[:i]` hoặc trong `s[i+1:]`. Đáp án của `s` = chuỗi dài hơn trong hai đáp án con.

**Luật sớm nhất:** mọi chuỗi con của `s[:i]` đều bắt đầu trước mọi chuỗi con của `s[i+1:]`. Khi hòa độ dài, chọn `left` là chọn chuỗi sớm hơn. Trong một phía, quy nạp đảm bảo đã chọn chuỗi sớm nhất.

**Trường hợp không có ký tự hỏng:** mọi ký tự đều có bạn trong `s` → bản thân `s` là nice và là chuỗi con dài nhất có thể.

Ta đối chiếu với brute force trên **mọi** chuỗi ngắn trên bảng chữ `{a, A, b, B}`.

In [19]:
from itertools import product

checked = 0
for n in range(1, 8):
    for tup in product("aAbB", repeat=n):
        t = "".join(tup)
        expected = longest_nice_brute(t)
        assert longest_nice_dc(t) == expected, t
        assert SolutionBitmask().longestNiceSubstring(t) == expected, t
        assert longest_nice_window(t) == expected, t
        checked += 1
print("Đối chiếu trên", checked, "chuỗi: chia để trị == bitmask == window == brute force")

Đối chiếu trên 21844 chuỗi: chia để trị == bitmask == window == brute force


## 10. Độ phức tạp

Gọi `n = len(s)`.

- **Brute force:** `n²/2` chuỗi con × `O(n)` (slice + set) → **Time `O(n³)`**, Space `O(n)`.
- **Bitmask:** `n²/2` cặp `(i, j)` × `O(1)` → **Time `O(n²)`**, **Space `O(1)`**.
- **Chia để trị:** mỗi tầng đệ quy, tổng độ dài các đoạn ≤ `n` và mỗi đoạn tốn `O(độ dài)` để dựng set + quét → `O(n)` mỗi tầng. Mỗi lần cắt bỏ đi ít nhất một ký tự nên độ sâu ≤ `n` → **Time `O(n²)`** xấu nhất. Trong thực tế, số chữ cái khác nhau giới hạn ở 26, nên độ sâu "có ích" thường nhỏ. **Space** `O(n)` cho stack đệ quy (cộng các slice).
- **Sliding window:** **Time `O(26 · n)`**.

Với `n ≤ 100`, tất cả đều chạy tức thì.

## 11. Bộ test

Nhóm test:

1. Ba ví dụ của đề.
2. Một ký tự; hai ký tự là cặp (`"aA"`) và không phải cặp (`"ab"`, `"AA"`).
3. Hòa độ dài → lấy sớm nhất: `"dDzeE"`, `"aAbBcxXcC"`.
4. Cả chuỗi nice: `"abABB"`.
5. Chuỗi nice nằm ở cuối / ở giữa.
6. Đối chiếu ngẫu nhiên giữa các lớp `Solution`.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS ({actual!r})")

sol = Solution()
check("đề 1", sol.longestNiceSubstring("YazaAay"), "aAa")
check("đề 2", sol.longestNiceSubstring("Bb"), "Bb")
check("đề 3", sol.longestNiceSubstring("c"), "")
check("cặp aA", sol.longestNiceSubstring("aA"), "aA")
check("không cặp ab", sol.longestNiceSubstring("ab"), "")
check("cùng kiểu AA", sol.longestNiceSubstring("AA"), "")
check("hòa -> sớm nhất", sol.longestNiceSubstring("dDzeE"), "dD")
check("cả chuỗi nice", sol.longestNiceSubstring("abABB"), "abABB")
check("nice ở cuối", sol.longestNiceSubstring("xyzQqWw"), "QqWw")
check("nice ở giữa", sol.longestNiceSubstring("kaAbBm"), "aAbB")
check("hòa nhiều đoạn", sol.longestNiceSubstring("aAxbBycC"), "aA")

đề 1                           PASS ('aAa')
đề 2                           PASS ('Bb')
đề 3                           PASS ('')
cặp aA                         PASS ('aA')
không cặp ab                   PASS ('')
cùng kiểu AA                   PASS ('')
hòa -> sớm nhất                PASS ('dD')
cả chuỗi nice                  PASS ('abABB')
nice ở cuối                    PASS ('QqWw')
nice ở giữa                    PASS ('aAbB')
hòa nhiều đoạn                 PASS ('aA')


In [21]:
import random

rng = random.Random(1763)
impls = [Solution(), SolutionBrute(), SolutionBitmask()]
for _ in range(400):
    n = rng.randint(1, 40)
    alphabet = rng.choice(["aAbB", "aAbBcC", "abcABCxyz", "aAzZqQ"])
    t = "".join(rng.choice(alphabet) for _ in range(n))
    outs = {x.longestNiceSubstring(t) for x in impls} | {longest_nice_window(t)}
    assert len(outs) == 1, (t, outs)
print("400 chuỗi ngẫu nhiên: mọi cách khớp nhau")

400 chuỗi ngẫu nhiên: mọi cách khớp nhau


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` khi cập nhật đáp án ở brute force

Chuỗi cùng độ dài xuất hiện **sau** sẽ ghi đè chuỗi sớm hơn.

In [22]:
def wrong_ge(s):
    best = ""
    for i in range(len(s)):
        for j in range(i, len(s)):
            t = s[i:j + 1]
            if is_nice(t) and len(t) >= len(best):
                best = t
    return best

print("sai:", repr(wrong_ge("dDzeE")), "| đúng:", repr(longest_nice_brute("dDzeE")))

sai: 'eE' | đúng: 'dD'


**Sửa:** `>` ở brute force; còn ở chia để trị thì ngược lại: `len(left) >= len(right)` để ưu tiên bên trái.

### Lỗi 2: Kiểm tra nice chỉ theo một chiều

Chỉ kiểm "mọi chữ thường có chữ hoa" mà quên "mọi chữ hoa có chữ thường".

In [23]:
def wrong_is_nice(t):
    chars = set(t)
    return all(c.upper() in chars for c in chars if c.islower())

print("'AB' sai:", wrong_is_nice("AB"), "| đúng:", is_nice("AB"))

'AB' sai: True | đúng: False


**Sửa:** dùng `swapcase()` cho mọi ký tự, hoặc so `lower_mask == upper_mask`.

### Lỗi 3: Chia để trị nhưng lấy `set` của **chuỗi gốc** thay vì đoạn hiện tại

Ký tự có bạn trong chuỗi gốc nhưng bạn nằm **ngoài** đoạn đang xét → không còn hợp lệ. Phải dựng `set` cho từng đoạn trong mỗi lần gọi đệ quy.

In [24]:
def wrong_dc(s, chars=None):
    if chars is None:
        chars = set(s)                 # chỉ tính một lần cho chuỗi gốc
    if len(s) < 2:
        return ""
    for i, ch in enumerate(s):
        if ch.swapcase() not in chars:
            l, r = wrong_dc(s[:i], chars), wrong_dc(s[i + 1:], chars)
            return l if len(l) >= len(r) else r
    return s

print("sai:", repr(wrong_dc("YazaAay")), "| đúng:", repr(longest_nice_dc("YazaAay")))

sai: 'aAay' | đúng: 'aAa'


Ở đây `'y'` và `'Y'` đều có trong chuỗi gốc nên sau khi cắt, `"aAay"` bị coi là nice dù `'Y'` đã nằm ngoài đoạn.

### Lỗi 4: Nhầm chuỗi con với dãy con

`"aXA"` có dãy con `"aA"` (bỏ `X`) nhưng **không** có chuỗi con nice — đáp án `""`.

In [25]:
print(repr(longest_nice_dc("aXA")))

''


## 13. Mẹo nhận dạng pattern

- **"Chuỗi con dài nhất thỏa điều kiện"** với `n` nhỏ (≤ 100) → brute force mọi `(i, j)` là hợp lệ; nâng cấp bằng **mở rộng dần + cập nhật trạng thái** (`O(n²)`).
- Có một phần tử **chắc chắn không thể nằm trong đáp án** → **chia để trị**: cắt tại đó, giải hai bên. (Cùng ý tưởng với bài 395.)
- Tập chữ cái nhỏ (26) → **bitmask** thay `set`.
- Luật "hòa thì lấy sớm nhất" → chú ý `>` vs `>=` theo thứ tự duyệt.

Bài cùng họ: 395 Longest Substring with At Least K Repeating Characters (chia để trị y hệt), 3 Longest Substring Without Repeating Characters, 2062 Count Vowel Substrings.

## 14. Biến thể

### 14.1. Đếm số chuỗi con nice (khác rỗng)

In [26]:
def count_nice_substrings(s):
    cnt = 0
    for i in range(len(s)):
        lower = upper = 0
        for j in range(i, len(s)):
            ch = s[j]
            if ch.islower():
                lower |= 1 << (ord(ch) - ord('a'))
            else:
                upper |= 1 << (ord(ch) - ord('A'))
            if lower == upper:
                cnt += 1
    return cnt

print(count_nice_substrings("YazaAay"))   # aA, Aa, aAa -> 3
print(count_nice_substrings("abAB"))

3
1


### 14.2. Bài 395 — chuỗi con dài nhất mà mọi ký tự xuất hiện ≥ k lần

Cùng khung chia để trị: ký tự có tổng số lần < k trong đoạn hiện tại là "ký tự hỏng".

In [27]:
from collections import Counter

def longest_substring_k(s, k):
    if len(s) < k:
        return 0
    freq = Counter(s)
    for i, ch in enumerate(s):
        if freq[ch] < k:
            return max(longest_substring_k(s[:i], k), longest_substring_k(s[i + 1:], k))
    return len(s)

print(longest_substring_k("aaabb", 3), longest_substring_k("ababbc", 2))   # 3, 5

3 5


### 14.3. Trả về **vị trí** (start, end) thay vì chuỗi

Dễ dàng với cách bitmask: đã lưu `best_i, best_len`.

In [28]:
def longest_nice_span(s):
    best_i, best_len = 0, 0
    for i in range(len(s)):
        lower = upper = 0
        for j in range(i, len(s)):
            ch = s[j]
            if ch.islower(): lower |= 1 << (ord(ch) - 97)
            else: upper |= 1 << (ord(ch) - 65)
            if lower == upper and j - i + 1 > best_len:
                best_i, best_len = i, j - i + 1
    return (best_i, best_i + best_len - 1) if best_len else (-1, -1)

print(longest_nice_span("YazaAay"), longest_nice_span("c"))

(3, 5) (-1, -1)


`97` và `65` lần lượt là `ord('a')` và `ord('A')`.

## 15. Bài tập tự luyện

Các hàm TODO không được gọi khi chạy notebook. Đáp án ở mục 17.

### Bài 1 — Tính tay

Các chuỗi sau có nice không: `"aAbB"`, `"aAb"`, `"AaAa"`, `"zZy"`? Với `s = "qQwWx"`, đáp án là gì?

### Bài 2 — Dự đoán output

`'q'.swapcase()`, `bin(1 << 3)`, `("abcde")[1:4]`, `set("aAa")` lần lượt ra gì?

### Bài 3 — Điền TODO: hàm `is_nice` bằng bitmask

In [29]:
def exercise_is_nice_mask(t: str) -> bool:
    # TODO 1: lower = upper = 0
    # TODO 2: chữ thường -> bật bit trong lower; chữ hoa -> bật bit trong upper
    # TODO 3: trả lower == upper
    pass

# assert exercise_is_nice_mask("abABB") == True
# assert exercise_is_nice_mask("abA") == False

### Bài 4 — Tự viết lời giải chia để trị

In [30]:
def exercise_longest_nice(s: str) -> str:
    # TODO: base case len < 2; tìm ký tự hỏng; đệ quy hai bên; ưu tiên bên trái khi hòa
    pass

# assert exercise_longest_nice("YazaAay") == "aAa"
# assert exercise_longest_nice("dDzeE") == "dD"

### Bài 5 — Biến thể: độ dài chuỗi con nice dài nhất **chỉ dùng chữ cái a–e**

Cho `s`, chỉ xét các chuỗi con **không chứa** chữ cái nào ngoài `a..e` / `A..E`. Gợi ý: các ký tự ngoài `a–e` là "ký tự hỏng" tự nhiên.

In [31]:
def exercise_longest_nice_ae(s: str) -> int:
    # TODO: cắt chuỗi tại các ký tự ngoài a-e, giải từng đoạn bằng hàm đã có
    pass

# assert exercise_longest_nice_ae("aAzZbB") == 2        # 'zZ' nice nhưng bị loại
# assert exercise_longest_nice_ae("aAbBfFcC") == 4

### Bài 6 — Mini challenge: chuỗi con nice dài nhất chứa **đúng 2** chữ cái khác nhau

Ví dụ `"aAbBcC"` → `"aAbB"` (2 chữ cái a, b). Không có thì `""`. Dùng brute force mở rộng dần, đếm số bit bật bằng `bin(mask).count("1")`.

In [32]:
def exercise_longest_nice_two_letters(s: str) -> str:
    # TODO: như bitmask, thêm điều kiện số chữ cái khác nhau == 2
    pass

# assert exercise_longest_nice_two_letters("aAbBcC") == "aAbB"
# assert exercise_longest_nice_two_letters("aAxyz") == ""

## 16. Gợi ý 3 mức

**Bài 1** — M1: xét từng chữ cái có trong chuỗi. M2: `"aAb"` thiếu `B`; `"zZy"` thiếu `Y`. M3: nice / không / nice / không; `"qQwWx"` → `"qQwW"`.

**Bài 2** — M1: `swapcase` đổi kiểu chữ. M2: `1 << 3 = 8`. M3: `'Q'`, `'0b1000'`, `'bcd'`, `{'a', 'A'}`.

**Bài 3** — M1: hai số nguyên làm tập. M2: `ord(ch) - ord('a')` cho chữ thường, `ord(ch) - ord('A')` cho chữ hoa. M3: `lower |= 1 << k`.

**Bài 4** — M1: chép khung mục 5. M2: `if ch.swapcase() not in chars`. M3: `return left if len(left) >= len(right) else right`.

**Bài 5** — M1: chuỗi nice thỏa đề không được chứa ký tự ngoài a–e. M2: duyệt `s`, gom các đoạn liên tiếp chỉ gồm a–e/A–E. M3: `max(len(longest_nice_dc(đoạn)) for đoạn in các_đoạn)`, nhớ trường hợp không có đoạn nào.

**Bài 6** — M1: giữ thêm điều kiện về số chữ cái. M2: `bin(lower | upper).count("1")` là số chữ cái khác nhau. M3: cập nhật khi `lower == upper` và số đó `== 2` và dài hơn đáp án.

## 17. Đáp án tham khảo

**Bài 1:** `"aAbB"` nice, `"aAb"` không, `"AaAa"` nice, `"zZy"` không. `"qQwWx"` → `"qQwW"`.

**Bài 2:** `'Q'`, `'0b1000'`, `'bcd'`, `{'a', 'A'}` (thứ tự in của set có thể khác).

In [33]:
print([is_nice(t) for t in ["aAbB", "aAb", "AaAa", "zZy"]], repr(longest_nice_dc("qQwWx")))
print('q'.swapcase(), bin(1 << 3), "abcde"[1:4], sorted(set("aAa")))

[True, False, True, False] 'qQwW'
Q 0b1000 bcd ['A', 'a']


In [34]:
def answer_is_nice_mask(t):
    lower = upper = 0
    for ch in t:
        if ch.islower():
            lower |= 1 << (ord(ch) - ord('a'))
        else:
            upper |= 1 << (ord(ch) - ord('A'))
    return lower == upper


def answer_longest_nice(s):
    if len(s) < 2:
        return ""
    chars = set(s)
    for i, ch in enumerate(s):
        if ch.swapcase() not in chars:
            left = answer_longest_nice(s[:i])
            right = answer_longest_nice(s[i + 1:])
            return left if len(left) >= len(right) else right
    return s


def answer_longest_nice_ae(s):
    best, seg = 0, []
    for ch in s + "#":                       # '#' làm "lính canh" để chốt đoạn cuối
        if ch.lower() in "abcde" and ch.isalpha():
            seg.append(ch)
        else:
            best = max(best, len(answer_longest_nice("".join(seg))))
            seg = []
    return best


def answer_longest_nice_two_letters(s):
    best_i, best_len = 0, 0
    for i in range(len(s)):
        lower = upper = 0
        for j in range(i, len(s)):
            ch = s[j]
            if ch.islower():
                lower |= 1 << (ord(ch) - ord('a'))
            else:
                upper |= 1 << (ord(ch) - ord('A'))
            if lower == upper and bin(lower).count("1") == 2 and j - i + 1 > best_len:
                best_i, best_len = i, j - i + 1
    return s[best_i:best_i + best_len]


assert answer_is_nice_mask("abABB") == True
assert answer_is_nice_mask("abA") == False
assert answer_longest_nice("YazaAay") == "aAa"
assert answer_longest_nice("dDzeE") == "dD"
assert answer_longest_nice_ae("aAzZbB") == 2
assert answer_longest_nice_ae("aAbBfFcC") == 4
assert answer_longest_nice_two_letters("aAbBcC") == "aAbB"
assert answer_longest_nice_two_letters("aAxyz") == ""
rng = random.Random(63)
for _ in range(300):
    t = "".join(rng.choice("aAbBcC") for _ in range(rng.randint(0, 12)))
    assert answer_is_nice_mask(t) == is_nice(t)
    assert answer_longest_nice(t) == longest_nice_brute(t)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def longestNiceSubstring(self, s: str) -> str:
        if len(s) < 2:
            return ""
        chars = set(s)                          # set của ĐOẠN HIỆN TẠI
        for i, ch in enumerate(s):
            if ch.swapcase() not in chars:      # ký tự hỏng -> cắt
                left = self.longestNiceSubstring(s[:i])
                right = self.longestNiceSubstring(s[i + 1:])
                return left if len(left) >= len(right) else right   # hòa -> bên trái
        return s
```

Template chia để trị "cắt tại phần tử hỏng":

```text
def solve(đoạn):
    if đoạn quá ngắn: return rỗng
    for vị trí x hỏng trong đoạn:
        return tốt_hơn(solve(trái của x), solve(phải của x))
    return đoạn   # không có phần tử hỏng
```

Checklist trước khi submit:

- [ ] Kiểm tra nice theo **cả hai chiều** (hoa ↔ thường)?
- [ ] `set` được dựng lại cho **từng đoạn** khi đệ quy?
- [ ] Hòa độ dài thì lấy chuỗi **sớm nhất** (`>=` cho bên trái ở chia để trị, `>` ở brute force)?
- [ ] Trả `""` khi không có chuỗi nice?
- [ ] Đã thử `"YazaAay"`, `"dDzeE"`, `"c"`, `"aXA"`?